# Actividad 01: Base de datos OLTP para el biobanco de Parasol Corp

Parasol Corp es una empresa de investigación farmacéutica y biotecnológica (a.k.a. Red Queen Data Labs). Su **biobanco** recibe muestras biológicas de donantes, las divide en **alícuotas**, las congela en ubicaciones controladas, registra cada movimiento y cada control de calidad, y proporciona alícuotas a los experimentos de los laboratorios.

**El problema:** todo eso se gestiona hoy en hojas de cálculo sueltas: no hay integridad referencial, ni auditoría de movimientos, ni forma de saber qué alícuota se usó en qué experimento.

**Tu trabajo:** diseñar la base de datos **OLTP** del biobanco con PK, FK, relaciones **1:N** y **N:M**, y cerrar experimentando con transacciones **ACID**.

Caso de negocio completo: [brief de Wesker](../BRIEF.md).

## El mapa del modelo

Estas son las **10 tablas** que vas a construir, con sus relaciones. No hace falta entenderlo todo ahora: vuelve a este mapa cada vez que termines un paso.

```mermaid
erDiagram
    DONANTES ||--o{ MUESTRAS : FK-donante
    USUARIOS ||--o{ MUESTRAS : FK-recibido-por
    MUESTRAS ||--o{ ALIQUOTAS : FK-muestra
    UBICACIONES ||--o{ ALIQUOTAS : FK-ubicacion
    ALIQUOTAS ||--o{ MOVIMIENTOS_MUESTRA : FK-aliquota
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-origen
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-destino
    USUARIOS ||--o{ MOVIMIENTOS_MUESTRA : FK-usuario
    ALIQUOTAS ||--o{ CONTROLES_CALIDAD : FK-aliquota
    USUARIOS ||--o{ CONTROLES_CALIDAD : FK-usuario
    LABORATORIOS ||--o{ EXPERIMENTOS : FK-laboratorio
    USUARIOS ||--o{ EXPERIMENTOS : FK-usuario
    EXPERIMENTOS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-experimento
    ALIQUOTAS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-aliquota
```


---

## Vocabulario: ¿qué es una alícuota?

Una **alícuota** es una porción en la que divides una muestra más grande para congelarla y usarla por separado. Ejemplo: un tubo de 50 ml de sangre (la *muestra*) se divide en 4 tubos de 5 ml (las *alícuotas*) - cada uno con su propia ubicación en el congelador, su estado y su historial.

Así se descongela solo lo que se necesita y la muestra madre queda protegida.

> **Recuerda:** una **Primary Key (PK)** identifica de forma única cada fila; una **Foreign Key (FK)** referencia la PK de otra tabla y garantiza la **integridad referencial**. Relaciones **1:N** = una fila de la tabla A se relaciona con muchas de la B.

In [2]:
import duckdb

con = duckdb.connect(':memory:')
print('Base de datos DuckDB en memoria creada.')

Base de datos DuckDB en memoria creada.


In [4]:
con.execute("SELECT VERSION();").df() 

,"""version""()"
0,v2.0.0


---

## Tablas de referencia (ya hechas)

Para centrarte en las relaciones, estas 4 tablas vienen con su DDL y sus datos. **Cópialas también a `scripts/schema.py`** al final (el entregable son las 10 tablas del modelo).

| Tabla | Qué guarda |
|-------|------------|
| `usuarios` | Quién opera el biobanco (Jill, Rebecca, Chris, Richard) |
| `laboratorios` | Los laboratorios de Parasol Corp |
| `donantes` | Origen de las muestras |
| `ubicaciones` | Posiciones de congelación (congelador, rack, posición) |

### Las 4 tablas que ya existen

Ya están creadas. Fíjate en sus claves: serán el destino de las FK que vas a crear.

```mermaid
erDiagram
    USUARIOS {
        int usuario_id PK
        string nombre
        string rol
    }
    LABORATORIOS {
        int laboratory_id PK
        string laboratory_name
        string country
    }
    DONANTES {
        int donante_id PK
        string tipo
        string pais
    }
    UBICACIONES {
        int ubicacion_id PK
        string congelador
        string posicion
    }
```


In [5]:
con.execute("""
    CREATE TABLE usuarios (
        usuario_id INTEGER PRIMARY KEY,
        nombre VARCHAR NOT NULL,
        rol VARCHAR,
        email VARCHAR
    );
""")

con.execute("""
    INSERT INTO usuarios VALUES
    (1, 'Jill Valentine', 'Data Analyst', 'jill@parasol.com'),
    (2, 'Rebecca Chambers', 'Data Scientist', 'rebecca@parasol.com'),
    (3, 'Chris Redfield', 'Data Engineer', 'chris@parasol.com'),
    (4, 'Richard Aiken', 'DBA', 'richard@parasol.com');
""")

In [6]:
con.execute("""
    CREATE TABLE laboratorios (
        laboratory_id INTEGER PRIMARY KEY,
        laboratory_name VARCHAR NOT NULL,
        facility_type VARCHAR,
        city VARCHAR,
        country VARCHAR,
        security_level INTEGER,
        opening_date DATE
    );
""")

con.execute("""
    INSERT INTO laboratorios VALUES
    (3, 'Arklay Research Lab', 'Underground Lab', 'Ringtail City', 'USA', 5, '1988-07-12'),
    (7, 'Tanuki BioPlant', 'Manufacturing Lab', 'Osaka', 'Japan', 3, '2002-11-30'),
    (9, 'Spencer R&D Annex', 'Research Lab', 'Geneva', 'Switzerland', 4, '1995-05-04'),
    (11, 'Hive Data Platform', 'Data Center', 'Berlin', 'Germany', 5, '2010-01-15'),
    (13, 'Coati Creek Outpost', 'Field Station', 'Coati Creek', 'USA', 2, '2015-09-09');
""")

In [7]:
con.execute("""
    CREATE TABLE donantes (
        donante_id INTEGER PRIMARY KEY,
        tipo VARCHAR NOT NULL,
        pais VARCHAR,
        fecha_alta DATE
    );
""")

con.execute("""
    INSERT INTO donantes VALUES
    (1, 'Humano', 'Spain', '2025-11-02'),
    (2, 'Humano', 'USA', '2025-12-15'),
    (3, 'Animal', 'Japan', '2026-01-10'),
    (4, 'Ambiente', 'Germany', '2026-02-01'),
    (5, 'Humano', 'Italy', '2026-03-05'),
    (6, 'Animal', 'Switzerland', '2026-03-20');
""")

In [8]:
con.execute("""
    CREATE TABLE ubicaciones (
        ubicacion_id INTEGER PRIMARY KEY,
        congelador VARCHAR,
        rack VARCHAR,
        posicion VARCHAR,
        temp_c INTEGER
    );
""")

con.execute("""
    INSERT INTO ubicaciones VALUES
    (1, 'Congelador A', 'Rack 1', 'Pos 1', -80),
    (2, 'Congelador A', 'Rack 1', 'Pos 2', -80),
    (3, 'Congelador A', 'Rack 2', 'Pos 1', -80),
    (4, 'Congelador B', 'Rack 1', 'Pos 1', -20),
    (5, 'Congelador B', 'Rack 1', 'Pos 2', -20),
    (6, 'Congelador C', 'Rack 1', 'Pos 1', 4);
""")

In [9]:
con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name;
""").df()

,table_name
0,donantes
1,laboratorios
2,ubicaciones
3,usuarios


---

## Paso 1: La tabla `muestras`

Cada muestra llega de **un** donante (relación **1:N**) y la recibe **un** usuario del biobanco.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| sample_id | INTEGER | PRIMARY KEY |
| donante_id | INTEGER | FOREIGN KEY -> donantes(donante_id) |
| sample_type | VARCHAR | NOT NULL |
| collection_date | DATE | |
| risk_level | VARCHAR | |
| status | VARCHAR | |
| received_by | INTEGER | FOREIGN KEY -> usuarios(usuario_id) |

Inserta estos datos:

| sample_id | donante_id | sample_type | collection_date | risk_level | status | received_by |
|-----------|------------|-------------|-----------------|------------|--------|-------------|
| 5510 | 1 | Blood | 2026-01-15 | High | Active | 1 |
| 5511 | 1 | Tissue | 2026-01-15 | High | Active | 1 |
| 5512 | 2 | Blood | 2026-02-02 | Medium | Active | 2 |
| 5513 | 3 | Saliva | 2026-02-18 | Low | Active | 2 |
| 5514 | 4 | Tissue | 2026-03-01 | Critical | Quarantine | 3 |
| 5515 | 5 | Blood | 2026-03-12 | Medium | Active | 3 |

### El modelo hasta este paso

Una muestra pertenece a **un** donante y la recibe **un** usuario.

```mermaid
erDiagram
    DONANTES ||--o{ MUESTRAS : FK-donante
    USUARIOS ||--o{ MUESTRAS : FK-recibido-por
    MUESTRAS {
        int sample_id PK
        int donante_id FK
        string sample_type
        date collection_date
        string risk_level
        string status
        int received_by FK
    }
```


In [10]:
con.execute("""
    CREATE TABLE muestras (
        sample_id INTEGER PRIMARY KEY,
        donante_id INTEGER,
        sample_type VARCHAR NOT NULL,
        collection_date DATE,
        risk_level VARCHAR,
        status VARCHAR,
        received_by INTEGER,

        FOREIGN KEY (donante_id) REFERENCES donantes(donante_id),
        FOREIGN KEY (received_by) REFERENCES usuarios(usuario_id)
    );
""")

In [11]:
con.execute("""
    INSERT INTO muestras VALUES
    (5510, 1, 'Blood', '2026-01-15', 'High', 'Active', 1),
    (5511, 1, 'Tissue', '2026-01-15', 'High', 'Active', 1),
    (5512, 2, 'Blood', '2026-02-02', 'Medium', 'Active', 2),
    (5513, 3, 'Saliva', '2026-02-18', 'Low', 'Active', 2),
    (5514, 4, 'Tissue', '2026-03-01', 'Critical', 'Quarantine', 3),
    (5515, 5, 'Blood', '2026-03-12', 'Medium', 'Active', 3);
""")

print("Tabla muestras creada y datos insertados correctamente.")

Tabla muestras creada y datos insertados correctamente.


In [12]:
con.execute("""
    SELECT *
    FROM muestras
    ORDER BY sample_id;
""").df()

,sample_id,donante_id,sample_type,collection_date,risk_level,status,received_by
0,5510,1,Blood,2026-01-15,High,Active,1
1,5511,1,Tissue,2026-01-15,High,Active,1
2,5512,2,Blood,2026-02-02,Medium,Active,2
3,5513,3,Saliva,2026-02-18,Low,Active,2
4,5514,4,Tissue,2026-03-01,Critical,Quarantine,3
5,5515,5,Blood,2026-03-12,Medium,Active,3


In [13]:
con.execute("""
    SELECT
        m.sample_id,
        m.sample_type,
        d.tipo AS tipo_donante,
        d.pais,
        u.nombre AS recibido_por
    FROM muestras m
    JOIN donantes d
        ON m.donante_id = d.donante_id
    JOIN usuarios u
        ON m.received_by = u.usuario_id
    ORDER BY m.sample_id;
""").df()

,sample_id,sample_type,tipo_donante,pais,recibido_por
0,5510,Blood,Humano,Spain,Jill Valentine
1,5511,Tissue,Humano,Spain,Jill Valentine
2,5512,Blood,Humano,USA,Rebecca Chambers
3,5513,Saliva,Animal,Japan,Rebecca Chambers
4,5514,Tissue,Ambiente,Germany,Chris Redfield
5,5515,Blood,Humano,Italy,Chris Redfield


## Paso 2: La tabla `aliquotas`

Una muestra se divide en **muchas** alícuotas (relación **1:N**). Cada alícuota vive en **una** ubicación. Añade el **CHECK** `volumen_ml > 0`: un volumen negativo no tiene sentido.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| aliquota_id | INTEGER | PRIMARY KEY |
| sample_id | INTEGER | FOREIGN KEY -> muestras(sample_id) |
| volumen_ml | INTEGER | CHECK (volumen_ml > 0) |
| estado | VARCHAR | |
| ubicacion_id | INTEGER | FOREIGN KEY -> ubicaciones(ubicacion_id) |

Inserta estos datos:

| aliquota_id | sample_id | volumen_ml | estado | ubicacion_id |
|-------------|-----------|------------|--------|--------------|
| 101 | 5510 | 12 | Activa | 1 |
| 102 | 5510 | 10 | Activa | 2 |
| 103 | 5511 | 8 | En uso | 3 |
| 104 | 5512 | 15 | Activa | 4 |
| 105 | 5513 | 6 | Activa | 5 |
| 106 | 5514 | 9 | Retenida | 1 |
| 107 | 5515 | 11 | Activa | 2 |

### El modelo hasta este paso

Una muestra se divide en **muchas** alícuotas y cada alícuota vive en **una** ubicación.

```mermaid
erDiagram
    MUESTRAS ||--o{ ALIQUOTAS : FK-muestra
    UBICACIONES ||--o{ ALIQUOTAS : FK-ubicacion
    ALIQUOTAS {
        int aliquota_id PK
        int sample_id FK
        int volumen_ml
        string estado
        int ubicacion_id FK
    }
```


In [14]:
con.execute("""
    CREATE TABLE aliquotas (
        aliquota_id INTEGER PRIMARY KEY,
        sample_id INTEGER,
        volumen_ml INTEGER CHECK (volumen_ml > 0),
        estado VARCHAR,
        ubicacion_id INTEGER,

        FOREIGN KEY (sample_id)
            REFERENCES muestras(sample_id),

        FOREIGN KEY (ubicacion_id)
            REFERENCES ubicaciones(ubicacion_id)
    );
""")

print("Tabla aliquotas creada correctamente.")


Tabla aliquotas creada correctamente.


In [15]:
con.execute("""
    INSERT INTO aliquotas VALUES
    (101, 5510, 12, 'Activa', 1),
    (102, 5510, 10, 'Activa', 2),
    (103, 5511, 8, 'En uso', 3),
    (104, 5512, 15, 'Activa', 4),
    (105, 5513, 6, 'Activa', 5),
    (106, 5514, 9, 'Retenida', 1),
    (107, 5515, 11, 'Activa', 2);
""")

print("7 alícuotas insertadas correctamente.")

7 alícuotas insertadas correctamente.


In [16]:
con.execute("""
    SELECT *
    FROM aliquotas
    ORDER BY aliquota_id;
""").df()

,aliquota_id,sample_id,volumen_ml,estado,ubicacion_id
0,101,5510,12,Activa,1
1,102,5510,10,Activa,2
2,103,5511,8,En uso,3
3,104,5512,15,Activa,4
4,105,5513,6,Activa,5
5,106,5514,9,Retenida,1
6,107,5515,11,Activa,2


In [17]:
try:
    con.execute("""
        INSERT INTO aliquotas
        VALUES (999, 5510, -5, 'Activa', 1);
    """)
except Exception as e:
    print("La restricción CHECK ha funcionado correctamente.")
    print("Error:", e)

La restricción CHECK ha funcionado correctamente.
Error: Constraint Error: CHECK constraint failed on table "aliquotas" with expression CHECK((volumen_ml > 0))


In [18]:
con.execute("""
    SELECT
        m.sample_id,
        m.sample_type,
        a.aliquota_id,
        a.volumen_ml,
        a.estado
    FROM muestras m
    JOIN aliquotas a
        ON m.sample_id = a.sample_id
    ORDER BY m.sample_id, a.aliquota_id;
""").df()

,sample_id,sample_type,aliquota_id,volumen_ml,estado
0,5510,Blood,101,12,Activa
1,5510,Blood,102,10,Activa
2,5511,Tissue,103,8,En uso
3,5512,Blood,104,15,Activa
4,5513,Saliva,105,6,Activa
5,5514,Tissue,106,9,Retenida
6,5515,Blood,107,11,Activa


---

## Paso 3: La tabla `movimientos_muestra` (libro de auditoría)

Cada vez que una alícuota entra al biobanco o cambia de ubicación queda registrado **quién**, **cuándo** y **de dónde a dónde**. Es un ledger: las filas solo se insertan, nunca se borran.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| movimiento_id | INTEGER | PRIMARY KEY |
| aliquota_id | INTEGER | FOREIGN KEY -> aliquotas(aliquota_id) |
| ubicacion_origen | INTEGER | FOREIGN KEY -> ubicaciones(ubicacion_id) (NULL = entrada) |
| ubicacion_destino | INTEGER | FOREIGN KEY -> ubicaciones(ubicacion_id) |
| usuario_id | INTEGER | FOREIGN KEY -> usuarios(usuario_id) |
| fecha | TIMESTAMP | NOT NULL |
| tipo | VARCHAR | |

Inserta estos datos:

| movimiento_id | aliquota_id | ubicacion_origen | ubicacion_destino | usuario_id | fecha | tipo |
|---------------|-------------|------------------|-------------------|------------|-------|------|
| 1 | 101 | NULL | 1 | 3 | 2026-01-16 10:00:00 | Entrada |
| 2 | 102 | NULL | 2 | 3 | 2026-01-16 10:05:00 | Entrada |
| 3 | 103 | 1 | 3 | 2 | 2026-02-20 09:30:00 | Traslado |
| 4 | 104 | NULL | 4 | 3 | 2026-02-03 11:00:00 | Entrada |
| 5 | 105 | NULL | 5 | 3 | 2026-02-25 08:45:00 | Entrada |
| 6 | 106 | NULL | 1 | 4 | 2026-03-02 14:00:00 | Entrada |

### El modelo hasta este paso

Cada movimiento registra la alícuota, de dónde viene, a dónde va y quién lo hace. Fíjate en las **dos FK** a `ubicaciones`.

```mermaid
erDiagram
    ALIQUOTAS ||--o{ MOVIMIENTOS_MUESTRA : FK-aliquota
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-origen
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-destino
    USUARIOS ||--o{ MOVIMIENTOS_MUESTRA : FK-usuario
    MOVIMIENTOS_MUESTRA {
        int movimiento_id PK
        int aliquota_id FK
        int ubicacion_origen FK
        int ubicacion_destino FK
        int usuario_id FK
        timestamp fecha
        string tipo
    }
```


In [19]:
con.execute("""
    CREATE TABLE movimientos_muestra (
        movimiento_id INTEGER PRIMARY KEY,
        aliquota_id INTEGER,
        ubicacion_origen INTEGER,
        ubicacion_destino INTEGER,
        usuario_id INTEGER,
        fecha TIMESTAMP NOT NULL,
        tipo VARCHAR,

        FOREIGN KEY (aliquota_id)
            REFERENCES aliquotas(aliquota_id),

        FOREIGN KEY (ubicacion_origen)
            REFERENCES ubicaciones(ubicacion_id),

        FOREIGN KEY (ubicacion_destino)
            REFERENCES ubicaciones(ubicacion_id),

        FOREIGN KEY (usuario_id)
            REFERENCES usuarios(usuario_id)
    );
""")

print("Tabla movimientos_muestra creada correctamente.")


Tabla movimientos_muestra creada correctamente.


In [20]:
con.execute("""
    INSERT INTO movimientos_muestra VALUES
    (1, 101, NULL, 1, 3, '2026-01-16 10:00:00', 'Entrada'),
    (2, 102, NULL, 2, 3, '2026-01-16 10:05:00', 'Entrada'),
    (3, 103, 1, 3, 2, '2026-02-20 09:30:00', 'Traslado'),
    (4, 104, NULL, 4, 3, '2026-02-03 11:00:00', 'Entrada'),
    (5, 105, NULL, 5, 3, '2026-02-25 08:45:00', 'Entrada'),
    (6, 106, NULL, 1, 4, '2026-03-02 14:00:00', 'Entrada');
""")

print("6 movimientos insertados correctamente.")

6 movimientos insertados correctamente.


In [21]:
con.execute("""
    SELECT *
    FROM movimientos_muestra
    ORDER BY movimiento_id;
""").df()

,movimiento_id,aliquota_id,ubicacion_origen,ubicacion_destino,usuario_id,fecha,tipo
0,1,101,<NA>,1,3,2026-01-16 10:00:00,Entrada
1,2,102,<NA>,2,3,2026-01-16 10:05:00,Entrada
2,3,103,1,3,2,2026-02-20 09:30:00,Traslado
3,4,104,<NA>,4,3,2026-02-03 11:00:00,Entrada
4,5,105,<NA>,5,3,2026-02-25 08:45:00,Entrada
5,6,106,<NA>,1,4,2026-03-02 14:00:00,Entrada


In [22]:
con.execute("""
    SELECT
        m.movimiento_id,
        m.aliquota_id,
        m.ubicacion_origen,
        m.ubicacion_destino,
        u.nombre AS responsable,
        m.fecha,
        m.tipo
    FROM movimientos_muestra AS m
    JOIN usuarios AS u
        ON m.usuario_id = u.usuario_id
    ORDER BY m.fecha;
""").df()

,movimiento_id,aliquota_id,ubicacion_origen,ubicacion_destino,responsable,fecha,tipo
0,1,101,<NA>,1,Chris Redfield,2026-01-16 10:00:00,Entrada
1,2,102,<NA>,2,Chris Redfield,2026-01-16 10:05:00,Entrada
2,4,104,<NA>,4,Chris Redfield,2026-02-03 11:00:00,Entrada
3,3,103,1,3,Rebecca Chambers,2026-02-20 09:30:00,Traslado
4,5,105,<NA>,5,Chris Redfield,2026-02-25 08:45:00,Entrada
5,6,106,<NA>,1,Richard Aiken,2026-03-02 14:00:00,Entrada


In [23]:
con.execute("""
    SELECT COUNT(*) AS total_movimientos
    FROM movimientos_muestra;
""").df()

,total_movimientos
0,6


## Paso 4: La tabla `controles_calidad`

Cada control de calidad se aplica a **una** alícuota y lo firma **un** usuario.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| control_id | INTEGER | PRIMARY KEY |
| aliquota_id | INTEGER | FOREIGN KEY -> aliquotas(aliquota_id) |
| parametro | VARCHAR | |
| resultado | VARCHAR | |
| fecha | DATE | |
| usuario_id | INTEGER | FOREIGN KEY -> usuarios(usuario_id) |

Inserta estos datos:

| control_id | aliquota_id | parametro | resultado | fecha | usuario_id |
|------------|-------------|-----------|-----------|-------|------------|
| 1 | 101 | Pureza | Pass | 2026-01-20 | 2 |
| 2 | 102 | Volumen | Pass | 2026-01-20 | 2 |
| 3 | 104 | Viabilidad | Fail | 2026-02-10 | 4 |
| 4 | 105 | Pureza | Pass | 2026-03-01 | 2 |
| 5 | 106 | Viabilidad | Pass | 2026-03-05 | 4 |

### El modelo hasta este paso

Un control se aplica a **una** alícuota y lo firma **un** usuario.

```mermaid
erDiagram
    ALIQUOTAS ||--o{ CONTROLES_CALIDAD : FK-aliquota
    USUARIOS ||--o{ CONTROLES_CALIDAD : FK-usuario
    CONTROLES_CALIDAD {
        int control_id PK
        int aliquota_id FK
        string parametro
        string resultado
        date fecha
        int usuario_id FK
    }
```


In [24]:
con.execute("""
    CREATE TABLE controles_calidad (
        control_id INTEGER PRIMARY KEY,
        aliquota_id INTEGER,
        parametro VARCHAR,
        resultado VARCHAR,
        fecha DATE,
        usuario_id INTEGER,

        FOREIGN KEY (aliquota_id)
            REFERENCES aliquotas(aliquota_id),

        FOREIGN KEY (usuario_id)
            REFERENCES usuarios(usuario_id)
    );
""")

print("Tabla controles_calidad creada correctamente.")


Tabla controles_calidad creada correctamente.


In [25]:
con.execute("""
    INSERT INTO controles_calidad VALUES
    (1, 101, 'Pureza', 'Pass', '2026-01-20', 2),
    (2, 102, 'Volumen', 'Pass', '2026-01-20', 2),
    (3, 104, 'Viabilidad', 'Fail', '2026-02-10', 4),
    (4, 105, 'Pureza', 'Pass', '2026-03-01', 2),
    (5, 106, 'Viabilidad', 'Pass', '2026-03-05', 4);
""")

print("5 controles de calidad insertados correctamente.")

5 controles de calidad insertados correctamente.


In [26]:
con.execute("""
    SELECT *
    FROM controles_calidad
    ORDER BY control_id;
""").df()

,control_id,aliquota_id,parametro,resultado,fecha,usuario_id
0,1,101,Pureza,Pass,2026-01-20,2
1,2,102,Volumen,Pass,2026-01-20,2
2,3,104,Viabilidad,Fail,2026-02-10,4
3,4,105,Pureza,Pass,2026-03-01,2
4,5,106,Viabilidad,Pass,2026-03-05,4


In [27]:
con.execute("""
    SELECT
        c.control_id,
        c.aliquota_id,
        a.sample_id,
        c.parametro,
        c.resultado,
        c.fecha,
        u.nombre AS responsable
    FROM controles_calidad c
    JOIN aliquotas a
        ON c.aliquota_id = a.aliquota_id
    JOIN usuarios u
        ON c.usuario_id = u.usuario_id
    WHERE c.resultado = 'Fail';
""").df()

,control_id,aliquota_id,sample_id,parametro,resultado,fecha,responsable
0,3,104,5512,Viabilidad,Fail,2026-02-10,Richard Aiken


In [28]:
con.execute("""
    SELECT
        COUNT(*) AS total_controles,
        SUM(CASE WHEN resultado = 'Pass' THEN 1 ELSE 0 END)
            AS controles_superados,
        ROUND(
            100.0 * SUM(CASE WHEN resultado = 'Pass' THEN 1 ELSE 0 END)
            / COUNT(*),
            2
        ) AS quality_pass_rate
    FROM controles_calidad;
""").df()

,total_controles,controles_superados,quality_pass_rate
0,5,4.0,80.0


---

## Paso 5: La tabla `experimentos`

Cada experimento se realiza en **un** laboratorio y lo dirige **un** usuario.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| experiment_id | INTEGER | PRIMARY KEY |
| laboratory_id | INTEGER | FOREIGN KEY -> laboratorios(laboratory_id) |
| usuario_id | INTEGER | FOREIGN KEY -> usuarios(usuario_id) |
| started_at | TIMESTAMP | |
| ended_at | TIMESTAMP | |
| status | VARCHAR | |
| cost_eur | DECIMAL(12,2) | |

Inserta estos datos:

| experiment_id | laboratory_id | usuario_id | started_at | ended_at | status | cost_eur |
|---------------|---------------|------------|------------|----------|--------|----------|
| 87201 | 3 | 1 | 2026-04-01 09:00 | 2026-04-01 11:30 | completed | 1200.00 |
| 87202 | 3 | 2 | 2026-04-02 09:00 | 2026-04-02 12:15 | completed | 1500.50 |
| 87203 | 7 | 1 | 2026-04-02 08:00 | 2026-04-02 09:30 | completed | 950.00 |
| 87204 | 7 | 3 | 2026-04-15 09:12 | 2026-04-15 11:48 | failed | 410.00 |
| 87205 | 9 | 2 | 2026-04-16 10:00 | 2026-04-16 14:00 | completed | 2200.00 |
| 87206 | 3 | 1 | 2026-04-18 09:00 | 2026-04-18 10:30 | in progress | 300.00 |

### El modelo hasta este paso

Un experimento se realiza en **un** laboratorio y lo dirige **un** usuario.

```mermaid
erDiagram
    LABORATORIOS ||--o{ EXPERIMENTOS : FK-laboratorio
    USUARIOS ||--o{ EXPERIMENTOS : FK-usuario
    EXPERIMENTOS {
        int experiment_id PK
        int laboratory_id FK
        int usuario_id FK
        timestamp started_at
        timestamp ended_at
        string status
        float cost_eur
    }
```


In [29]:
con.execute("""
    CREATE TABLE experimentos (
        experiment_id INTEGER PRIMARY KEY,
        laboratory_id INTEGER,
        usuario_id INTEGER,
        started_at TIMESTAMP,
        ended_at TIMESTAMP,
        status VARCHAR,
        cost_eur DECIMAL(12,2),

        FOREIGN KEY (laboratory_id)
            REFERENCES laboratorios(laboratory_id),

        FOREIGN KEY (usuario_id)
            REFERENCES usuarios(usuario_id)
    );
""")

print("Tabla experimentos creada correctamente.")


Tabla experimentos creada correctamente.


In [30]:
con.execute("""
    INSERT INTO experimentos VALUES
    (87201, 3, 1, '2026-04-01 09:00', '2026-04-01 11:30', 'completed', 1200.00),
    (87202, 3, 2, '2026-04-02 09:00', '2026-04-02 12:15', 'completed', 1500.50),
    (87203, 7, 1, '2026-04-02 08:00', '2026-04-02 09:30', 'completed', 950.00),
    (87204, 7, 3, '2026-04-15 09:12', '2026-04-15 11:48', 'failed', 410.00),
    (87205, 9, 2, '2026-04-16 10:00', '2026-04-16 14:00', 'completed', 2200.00),
    (87206, 3, 1, '2026-04-18 09:00', '2026-04-18 10:30', 'in progress', 300.00);
""")

print("6 experimentos insertados correctamente.")

6 experimentos insertados correctamente.


In [31]:
con.execute("""
    SELECT
        laboratory_id,
        COUNT(*) AS total_experimentos,
        SUM(cost_eur) AS coste_total,
        ROUND(AVG(cost_eur), 2) AS coste_medio
    FROM experimentos
    GROUP BY laboratory_id
    ORDER BY laboratory_id;
""").df()

,laboratory_id,total_experimentos,coste_total,coste_medio
0,3,3,3000.5,1000.17
1,7,2,1360.0,680.00
2,9,1,2200.0,2200.00


In [32]:
con.execute("""
    SELECT
        status,
        COUNT(*) AS total
    FROM experimentos
    GROUP BY status
    ORDER BY status;
""").df()

,status,total
0,completed,4
1,failed,1
2,in progress,1


## Paso 6: La tabla `experimento_aliquotas` (relación N:M)

> **Recuerda:** una relación **N:M** (muchos a muchos) necesita una **tabla puente**: dos FK que juntas forman la PK compuesta.

Un experimento usa **muchas** alícuotas y una alícuota puede usarse en **muchos** experimentos -> **N:M**. Observa en los datos: el experimento `87201` usa las alícuotas 101 y 102, y la alícuota `101` se usa en los experimentos 87201 y 87205.

| Columna | Tipo | Restricción |
|---------|------|-------------|
| experiment_id | INTEGER | FOREIGN KEY -> experimentos(experiment_id) + PK compuesta |
| aliquota_id | INTEGER | FOREIGN KEY -> aliquotas(aliquota_id) + PK compuesta |
| cantidad_ml | INTEGER | |

Inserta estos datos:

| experiment_id | aliquota_id | cantidad_ml |
|---------------|-------------|-------------|
| 87201 | 101 | 12 |
| 87201 | 102 | 10 |
| 87202 | 103 | 8 |
| 87203 | 104 | 15 |
| 87204 | 105 | 6 |
| 87205 | 101 | 5 |
| 87205 | 106 | 9 |
| 87206 | 105 | 8 |

### La tabla puente de la relación N:M

La relación **N:M** se resuelve con una tabla puente: sus dos FK juntas forman la **PK compuesta**.

```mermaid
erDiagram
    EXPERIMENTOS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-experimento
    ALIQUOTAS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-aliquota
    EXPERIMENTO_ALIQUOTAS {
        int experiment_id PK
        int aliquota_id PK
        int cantidad_ml
    }
```


In [33]:
con.execute("""
    CREATE TABLE experimento_aliquotas (
        experiment_id INTEGER,
        aliquota_id INTEGER,
        cantidad_ml INTEGER,

        PRIMARY KEY (experiment_id, aliquota_id),

        FOREIGN KEY (experiment_id)
            REFERENCES experimentos(experiment_id),

        FOREIGN KEY (aliquota_id)
            REFERENCES aliquotas(aliquota_id)
    );
""")

print("Tabla experimento_aliquotas creada correctamente.")

Tabla experimento_aliquotas creada correctamente.


In [34]:
con.execute("""
    INSERT INTO experimento_aliquotas VALUES
    (87201, 101, 12),
    (87201, 102, 10),
    (87202, 103, 8),
    (87203, 104, 15),
    (87204, 105, 6),
    (87205, 101, 5),
    (87205, 106, 9),
    (87206, 105, 8);
""")

print("8 relaciones experimento-alícuota insertadas correctamente.")

8 relaciones experimento-alícuota insertadas correctamente.


In [35]:
con.execute("""
    SELECT *
    FROM experimento_aliquotas
    ORDER BY experiment_id, aliquota_id;
""").df()

,experiment_id,aliquota_id,cantidad_ml
0,87201,101,12
1,87201,102,10
2,87202,103,8
3,87203,104,15
4,87204,105,6
5,87205,101,5
6,87205,106,9
7,87206,105,8


In [36]:
try:
    con.execute("""
        INSERT INTO experimento_aliquotas
        VALUES (87201, 101, 3);
    """)
except Exception as e:
    print("La clave primaria compuesta funciona correctamente.")
    print("Error:", e)

La clave primaria compuesta funciona correctamente.
Error: Constraint Error: Duplicate key "experiment_id: 87201, aliquota_id: 101" violates primary key constraint.


In [37]:
con.execute("""
    SELECT
        e.experiment_id,
        e.status,
        a.aliquota_id,
        ea.cantidad_ml
    FROM experimentos e
    JOIN experimento_aliquotas ea
        ON ea.experiment_id = e.experiment_id
    JOIN aliquotas a
        ON a.aliquota_id = ea.aliquota_id
    ORDER BY e.experiment_id, a.aliquota_id;
""").df()

,experiment_id,status,aliquota_id,cantidad_ml
0,87201,completed,101,12
1,87201,completed,102,10
2,87202,completed,103,8
3,87203,completed,104,15
4,87204,failed,105,6
5,87205,completed,101,5
6,87205,completed,106,9
7,87206,in progress,105,8


---

## Verificación: tu modelo OLTP

Comprueba las 10 tablas y siente las relaciones con JOINs:

### Tu modelo completo

Así queda tu modelo tras los 6 pasos. Comprueba que coincide con tus `CREATE TABLE`.

```mermaid
erDiagram
    DONANTES ||--o{ MUESTRAS : FK-donante
    USUARIOS ||--o{ MUESTRAS : FK-recibido-por
    MUESTRAS ||--o{ ALIQUOTAS : FK-muestra
    UBICACIONES ||--o{ ALIQUOTAS : FK-ubicacion
    ALIQUOTAS ||--o{ MOVIMIENTOS_MUESTRA : FK-aliquota
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-origen
    UBICACIONES ||--o{ MOVIMIENTOS_MUESTRA : FK-destino
    USUARIOS ||--o{ MOVIMIENTOS_MUESTRA : FK-usuario
    ALIQUOTAS ||--o{ CONTROLES_CALIDAD : FK-aliquota
    USUARIOS ||--o{ CONTROLES_CALIDAD : FK-usuario
    LABORATORIOS ||--o{ EXPERIMENTOS : FK-laboratorio
    USUARIOS ||--o{ EXPERIMENTOS : FK-usuario
    EXPERIMENTOS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-experimento
    ALIQUOTAS ||--o{ EXPERIMENTO_ALIQUOTAS : FK-aliquota
```


In [38]:
def verify_schema():
    tables = [t[0] for t in con.execute('SHOW TABLES').fetchall()]
    print(f'Tablas: {tables}')
    required = ['usuarios', 'laboratorios', 'donantes', 'ubicaciones',
                'muestras', 'aliquotas', 'movimientos_muestra',
                'controles_calidad', 'experimentos', 'experimento_aliquotas']
    faltan = [t for t in required if t not in tables]
    print('Esquema OK' if not faltan else f'FALTAN: {faltan}')
    return tables

verify_schema()

Tablas: ['aliquotas', 'controles_calidad', 'donantes', 'experimento_aliquotas', 'experimentos', 'laboratorios', 'movimientos_muestra', 'muestras', 'ubicaciones', 'usuarios']
Esquema OK


['aliquotas',
 'controles_calidad',
 'donantes',
 'experimento_aliquotas',
 'experimentos',
 'laboratorios',
 'movimientos_muestra',
 'muestras',
 'ubicaciones',
 'usuarios']

In [39]:
# 1:N - cada muestra con sus alícuotas
con.execute("""
    SELECT m.sample_id, m.sample_type, a.aliquota_id, a.volumen_ml, a.estado
    FROM muestras m
    JOIN aliquotas a ON a.sample_id = m.sample_id
    ORDER BY m.sample_id, a.aliquota_id
""").fetchall()

[(5510, 'Blood', 101, 12, 'Activa'),
 (5510, 'Blood', 102, 10, 'Activa'),
 (5511, 'Tissue', 103, 8, 'En uso'),
 (5512, 'Blood', 104, 15, 'Activa'),
 (5513, 'Saliva', 105, 6, 'Activa'),
 (5514, 'Tissue', 106, 9, 'Retenida'),
 (5515, 'Blood', 107, 11, 'Activa')]

In [40]:
# N:M - experimentos con sus alícuotas (a través de la puente)
con.execute("""
    SELECT e.experiment_id, e.status, a.aliquota_id, ea.cantidad_ml
    FROM experimentos e
    JOIN experimento_aliquotas ea ON ea.experiment_id = e.experiment_id
    JOIN aliquotas a ON a.aliquota_id = ea.aliquota_id
    ORDER BY e.experiment_id, a.aliquota_id
""").fetchall()

[(87201, 'completed', 101, 12),
 (87201, 'completed', 102, 10),
 (87202, 'completed', 103, 8),
 (87203, 'completed', 104, 15),
 (87204, 'failed', 105, 6),
 (87205, 'completed', 101, 5),
 (87205, 'completed', 106, 9),
 (87206, 'in progress', 105, 8)]

---

## Paso 7: Transacciones ACID

> **Recuerda:** **ACID** = Atomicidad, Consistencia, Isolamiento y Durabilidad: las garantías de una transacción OLTP. O se aplican todas las operaciones de la transacción, o no se aplica ninguna.

Vamos a demostrar cada propiedad sobre el propio biobanco. La alícuota `107` es la única que todavía no está asignada a ningún experimento: la usaremos para las demos.

### Tarea 7.1: Mover una alícuota (transacción completa)

Mueve la alícuota `107` de la ubicación 2 a la 3: una transacción que actualiza la ubicación de la alícuota **y** registra el movimiento en el ledger. Ambas cosas o ninguna.

### Qué es una transacción

Las operaciones de una transacción se confirman **todas juntas** (COMMIT) o se deshacen **todas juntas** (ROLLBACK).

```mermaid
flowchart LR
    A[BEGIN] --> B[UPDATE de aliquotas]
    B --> C[INSERT en movimientos_muestra]
    C --> D{"¿Todo correcto?"}
    D -- si --> E[COMMIT]
    D -- no --> F[ROLLBACK]
```


In [41]:
con.execute("""
    SELECT aliquota_id, ubicacion_id, estado
    FROM aliquotas
    WHERE aliquota_id = 107;
""").df()

,aliquota_id,ubicacion_id,estado
0,107,2,Activa


In [42]:
try:
    # Iniciar la transacción
    con.execute("BEGIN TRANSACTION;")

    # Actualizar la ubicación de la alícuota
    con.execute("""
        UPDATE aliquotas
        SET ubicacion_id = 3
        WHERE aliquota_id = 107;
    """)

    # Registrar el movimiento en el ledger
    con.execute("""
        INSERT INTO movimientos_muestra
        VALUES (
            7,
            107,
            2,
            3,
            1,
            CURRENT_TIMESTAMP,
            'Traslado'
        );
    """)

    # Confirmar ambas operaciones
    con.execute("COMMIT;")

    print("Transacción completada correctamente.")

except Exception as e:
    con.execute("ROLLBACK;")
    print("Transacción revertida:", e)


Transacción completada correctamente.


In [43]:
con.execute("""
    SELECT
        a.aliquota_id,
        a.ubicacion_id,
        m.ubicacion_origen,
        m.ubicacion_destino,
        m.fecha,
        m.tipo
    FROM aliquotas a
    JOIN movimientos_muestra m
        ON a.aliquota_id = m.aliquota_id
    WHERE a.aliquota_id = 107;
""").df()

,aliquota_id,ubicacion_id,ubicacion_origen,ubicacion_destino,fecha,tipo
0,107,3,2,3,2026-11-02 16:09:02.275490,Traslado


### Tarea 7.2: Atomicidad - ROLLBACK total

Intenta mover la alícuota `107` a una ubicación **inexistente** (999). El primer paso de la transacción es válido, pero el segundo falla por la FK (`ubicacion_destino = 999` no existe). La transacción entera debe deshacerse: **ni el cambio de estado ni el movimiento** deben quedar registrados.

In [44]:
try:
    # 1. Iniciar la transacción
    con.execute("BEGIN TRANSACTION;")

    # 2. Primera operación: válida
    con.execute("""
        UPDATE aliquotas
        SET estado = 'Bloqueada'
        WHERE aliquota_id = 107;
    """)

    print("UPDATE ejecutado correctamente.")

    # 3. Segunda operación: provocamos un error
    con.execute("""
        INSERT INTO movimientos_muestra
        VALUES (
            8,
            107,
            3,
            999,
            1,
            CURRENT_TIMESTAMP,
            'Traslado'
        );
    """)

    # Solo se ejecutaría si ambas operaciones funcionaran
    con.execute("COMMIT;")

except Exception as e:
    print("Error detectado:", e)

    # 4. Revertir toda la transacción
    con.execute("ROLLBACK;")

    print("ROLLBACK ejecutado correctamente.")


UPDATE ejecutado correctamente.
Error detectado: Constraint Error: Violates foreign key constraint because key "ubicacion_id: 999" does not exist in the referenced table
ROLLBACK ejecutado correctamente.


In [45]:
con.execute("""
    SELECT
        aliquota_id,
        ubicacion_id,
        estado
    FROM aliquotas
    WHERE aliquota_id = 107;
""").df()

,aliquota_id,ubicacion_id,estado
0,107,3,Activa


In [46]:
con.execute("""
    SELECT *
    FROM movimientos_muestra
    WHERE movimiento_id = 8;
""").df()

,movimiento_id,aliquota_id,ubicacion_origen,ubicacion_destino,usuario_id,fecha,tipo


In [47]:
con.execute("""
    SELECT COUNT(*) AS total_movimientos
    FROM movimientos_muestra;
""").df()

,total_movimientos
0,7


### Tarea 7.3: Consistencia - CHECK constraint

Intenta crear una alícuota con `volumen_ml = -5`. El **CHECK** `volumen_ml > 0` debe rechazarla: la BD nunca pasa a un estado inválido.

In [48]:
try:
    con.execute("""
        INSERT INTO aliquotas (
            aliquota_id,
            sample_id,
            volumen_ml,
            estado,
            ubicacion_id
        )
        VALUES (
            999,
            5510,
            -5,
            'Activa',
            1
        );
    """)

    print("Alícuota insertada correctamente.")

except Exception as e:
    print("La base de datos ha rechazado la operación.")
    print("Motivo:", e)


La base de datos ha rechazado la operación.
Motivo: Constraint Error: CHECK constraint failed on table "aliquotas" with expression CHECK((volumen_ml > 0))


In [49]:
con.execute("""
    SELECT *
    FROM aliquotas
    WHERE aliquota_id = 999;
""").df()

,aliquota_id,sample_id,volumen_ml,estado,ubicacion_id


In [50]:
con.execute("""
    SELECT COUNT(*) AS total_aliquotas
    FROM aliquotas;
""").df()

,total_aliquotas
0,7


### Tarea 7.4: Isolamiento - transacciones concurrentes

Abre una segunda conexión (`con.cursor()`). Con una transacción abierta que cambia el estado de la alícuota `105`, la otra conexión debe seguir viendo el valor **anterior** hasta el COMMIT.

In [51]:
con.execute("""
    SELECT aliquota_id, estado
    FROM aliquotas
    WHERE aliquota_id = 105;
""").df()

,aliquota_id,estado
0,105,Activa


In [52]:
con2 = con.cursor()

print("Segunda conexión creada correctamente.")

Segunda conexión creada correctamente.


In [53]:
# Iniciamos la transacción en la conexión principal
con.execute("BEGIN TRANSACTION;")

# Modificamos el estado de la alícuota 105
con.execute("""
    UPDATE aliquotas
    SET estado = 'Bloqueada'
    WHERE aliquota_id = 105;
""")

print("Estado modificado, pero todavía sin COMMIT.")


Estado modificado, pero todavía sin COMMIT.


In [54]:
print("CONEXIÓN 1 (con):")

display(con.execute("""
    SELECT aliquota_id, estado
    FROM aliquotas
    WHERE aliquota_id = 105;
""").df())

print("CONEXIÓN 2 (con2):")

display(con2.execute("""
    SELECT aliquota_id, estado
    FROM aliquotas
    WHERE aliquota_id = 105;
""").df())

CONEXIÓN 1 (con):


,aliquota_id,estado
0,105,Bloqueada


CONEXIÓN 2 (con2):


,aliquota_id,estado
0,105,Activa


### Tarea 7.5: Durabilidad - el COMMIT es permanente

Tras un COMMIT, el cambio persiste: una nueva consulta lo ve aunque hubiéramos cerrado la conexión.

In [ ]:
# TODO: Comprueba que el cambio de la tarea anterior (estado = 'En uso')
#       persiste tras el COMMIT


### Tarea 7.6: Consultas OLTP típicas

Ejecuta las consultas características de un sistema OLTP: búsqueda por PK, INSERT de una fila y UPDATE de una fila.

In [ ]:
# TODO: Ejecuta una búsqueda por PK, un INSERT y un UPDATE sobre muestras


---

## Guarda tu trabajo

Copia **todas** tus sentencias `CREATE TABLE` e `INSERT INTO` (las 4 tablas de referencia + las 6 que has diseñado = **10 tablas**) en `scripts/schema.py`.

El fichero debe ser un script de Python independiente que:
1. Se conecta a DuckDB
2. Crea las 10 tablas
3. Inserta los datos de ejemplo

```bash
pytest tests/ -v
```

---

## Qué has hecho

1. Has diseñado el OLTP del biobanco: **10 tablas** con PK, FK y restricciones.
2. Has modelado relaciones **1:N** (donante->muestras->alícuotas) y una **N:M** con tabla puente (experimentos<->alícuotas).
3. Has creado un **ledger de auditoría** (`movimientos_muestra`) que registra cada movimiento.
4. Has experimentado con transacciones **ACID**: mover una alícuota, ROLLBACK por FK, CHECK de consistencia, isolamiento entre conexiones y durabilidad.

### Siguiente: Actividad 02: Normalización

Ahora que sabes diseñar un modelo OLTP, vamos a **normalizar** una tabla desnormalizada para reducir la redundancia.